In [237]:
# SUBGRAPH_URL = "https://gateway.thegraph.com/api/subgraphs/id/2z5Mn4WW7K4yR1iH9KdignREkTq9EM1S4GX3yLaztRFg"
# SILO_LENS = "0xC0e1bcFB1Ed68688B0d589A6807d05cF2D68b22b"
# FILE_NAME = "mainnet_borrowers.csv"
# RPC_URL = "https://eth.llamarpc.com"

# SUBGRAPH_URL = "https://gateway.thegraph.com/api/subgraphs/id/6NLL9WmjPYima4NhUpNEWeDu5eBXFuhP9QheRXkoJXR5"
# SILO_LENS = "0xA0380d22A4Ee658e9706b390ddf9646f184dd521"
# FILE_NAME = "avax_borrowers.csv"
# RPC_URL = "https://avalanche-c-chain-rpc.publicnode.com"

# SUBGRAPH_URL = "https://gateway.thegraph.com/api/subgraphs/id/DK5qWsSJSqkeW2GHDQQCB7xHnHwVN3K1LPpP6CYNXMh8"
# SILO_LENS = "0xF0B0218153633e6154c201d5A5d81128B0539336"
# FILE_NAME = "arb_borrowers.csv"
# RPC_URL = "https://arbitrum.drpc.org"

SUBGRAPH_URL = "https://gateway.thegraph.com/api/subgraphs/id/8wcbzcdNirQvk1ETh25wpVzb5GWs8DvugpbwrYnTCcxj"
SILO_LENS = "0xB95AD415b0fcE49f84FbD5B26b14ec7cf4822c69"
FILE_NAME = "sonic_borrowers.csv"
RPC_URL = "https://rpc.soniclabs.com"


In [238]:
import requests
import os
from typing import Dict, List, Any

def fetch_debt_positions(url: str, headers: Dict[str, str], batch_size: int = 1000) -> List[Dict[str, Any]]:
    """
    Fetch all positions from GraphQL API with automatic pagination.
    
    Args:
        url: GraphQL endpoint URL
        headers: Request headers including authorization
        batch_size: Number of items per request (default: 1000)
    
    Returns:
        List of all positions
    """
    all_positions = []
    skip = 0
    query_count = 0
    
    while True:
        # GraphQL query with pagination
        # Option 1: Use .format() - cleaner
        query = """
        {{
          positions(first: {batch_size}, skip: {skip}, where: {{ dTokenBalance_gt: 0 }}) {{
              id
              account {{
                id
              }}
              market {{
                id
                silo {{
                  id
                  market1 {{
                    id
                    name
                    inputToken {{
                      id
                      symbol
                      decimals
                    }}
                    sToken {{
                      id
                      symbol
                      decimals
                    }}
                    spToken {{
                      id
                      symbol
                      decimals
                    }}
                    dToken {{
                      id
                      symbol
                      decimals
                    }}
                  }}
                  market2 {{
                    id
                    name
                    inputToken {{
                      id
                      symbol
                      decimals
                    }}
                    sToken {{
                      id
                      symbol
                      decimals
                    }}
                    spToken {{
                      id
                      symbol
                      decimals
                    }}
                    dToken {{
                      id
                      symbol
                      decimals
                    }}
                  }}
                }}
              }}
              sTokenBalance
              spTokenBalance
              dTokenBalance
          }}
        }}
        """.format(batch_size=batch_size, skip=skip)
        
        # Make the request
        response = requests.post(
            url,
            json={'query': query},
            headers=headers
        )
        
        # Check response
        if response.status_code != 200:
            print(f"Error: {response.status_code}")
            print(response.text)
            break
        
        data = response.json()
        
        # Check for GraphQL errors
        if 'errors' in data:
            print(f"GraphQL errors: {data['errors']}")
            break
        
        positions = data.get('data', {}).get('positions', [])
        
        if not positions:
            # No more data
            break
        
        all_positions.extend(positions)
        query_count += 1
        
        print(f"Batch {query_count}: Fetched {len(positions)} positions (Total: {len(all_positions)})")
        
        # If we got fewer than batch_size items, we've reached the end
        if len(positions) < batch_size:
            break
        
        # Update skip for next batch
        skip = batch_size * query_count
    
    return all_positions


def fetch_lender_positions(url: str, headers: Dict[str, str], ids: List[str], batch_size: int = 1000) -> List[Dict[str, Any]]:
    all_positions = []
    
    # Process IDs in batches
    for i in range(0, len(ids), batch_size):
        batch_ids = ids[i:i + batch_size]
        batch_number = (i // batch_size) + 1
        
        # Format IDs as a GraphQL array: ["id1", "id2", "id3"]
        ids_string = '[' + ', '.join(f'"{id_}"' for id_ in batch_ids) + ']'
        
        # GraphQL query for this batch of IDs
        query = """
        {{
          positions(first:1000, where: {{ id_in: {ids} }}) {{
              id
              account {{
                id
              }}
              market {{
                id
                silo {{
                  id
                  market1 {{
                    id
                    name
                    inputToken {{
                      id
                      symbol
                      decimals
                    }}
                    sToken {{
                      id
                      symbol
                      decimals
                    }}
                    spToken {{
                      id
                      symbol
                      decimals
                    }}
                    dToken {{
                      id
                      symbol
                      decimals
                    }}
                  }}
                  market2 {{
                    id
                    name
                    inputToken {{
                      id
                      symbol
                      decimals
                    }}
                    sToken {{
                      id
                      symbol
                      decimals
                    }}
                    spToken {{
                      id
                      symbol
                      decimals
                    }}
                    dToken {{
                      id
                      symbol
                      decimals
                    }}
                  }}
                }}
              }}
              sTokenBalance
              spTokenBalance
              dTokenBalance
          }}
        }}
        """.format(ids=ids_string)
        
        # Make the request
        response = requests.post(
            url,
            json={'query': query},
            headers=headers
        )
        
        # Check response
        if response.status_code != 200:
            print(f"Error in batch {batch_number}: {response.status_code}")
            print(response.text)
            continue
        
        data = response.json()
        
        # Check for GraphQL errors
        if 'errors' in data:
            print(f"GraphQL errors in batch {batch_number}: {data['errors']}")
            continue
        
        positions = data.get('data', {}).get('positions', [])
        all_positions.extend(positions)
        
        print(f"Batch {batch_number}: Fetched {len(positions)} positions (Total: {len(all_positions)})")
    
    return all_positions


In [239]:
# Headers
headers = {
    'Content-Type': 'application/json',
    'Authorization': f'Bearer c5730755d734d94d0b03d1a024670c23'
}

# Fetch all positions
print("Fetching positions...")
debt_positions = fetch_debt_positions(SUBGRAPH_URL, headers)

print(f"\nSuccessfully fetched {len(debt_positions)} debt positions in total!")

Fetching positions...
Batch 1: Fetched 1000 positions (Total: 1000)
Batch 2: Fetched 1000 positions (Total: 2000)
Batch 3: Fetched 130 positions (Total: 2130)

Successfully fetched 2130 debt positions in total!


In [240]:
# FLATTEN GRAPHQL DATA FUNCTION

import pandas as pd
from typing import Any, Dict, List
import re

import pandas as pd
from typing import Any, Dict, List
import re

def flatten_graphql_to_dataframe(data: Dict[str, Any], is_collateral: bool,entity_key: str = 'positions') -> pd.DataFrame:
    """
    Convert GraphQL response to a flattened pandas DataFrame with special market processing.
    
    Identifies debt and collateral markets based on root market_id.
    - Debt market: Uses d_token (debt_token) and input_token (debt_asset)
    - Collateral market: Uses sp_token (collateral_token) and input_token (collateral_asset)
    """
    
    def camel_to_snake(name: str) -> str:
        """Convert camelCase to snake_case."""
        name = re.sub('(.)([A-Z][a-z]+)', r'\1_\2', name)
        return re.sub('([a-z0-9])([A-Z])', r'\1_\2', name).lower()
    
    def convert_to_numeric(value: Any) -> Any:
        """Convert string values to appropriate numeric types."""
        if not isinstance(value, str):
            return value
        
        if value.startswith('0x') or value.startswith('0X'):
            return value
        
        try:
            if '.' not in value and 'e' not in value.lower():
                return int(value)
        except (ValueError, TypeError):
            pass
        
        try:
            return float(value)
        except (ValueError, TypeError):
            pass
        
        return value
    
    def process_position(position: Dict[str, Any]) -> Dict[str, Any]:
        """Process a single position with market logic."""
        result = {
            'id': position.get('id'),
            'account_id': position.get('account', {}).get('id'),
            # 's_token_balance': position.get('sTokenBalance'),
            # 'sp_token_balance': position.get('spTokenBalance'),
            'd_token_balance': position.get('dTokenBalance'),
        }

        if is_collateral:
            result = {
                'id': position.get('id'),
                'account_id': position.get('account', {}).get('id'),
                's_token_balance': position.get('sTokenBalance'),
                'sp_token_balance': position.get('spTokenBalance'),
            }
        
        market = position.get('market', {})
        market_id = market.get('id')
        result['market_id'] = market_id
        
        silo = market.get('silo', {})
        result['silo_id'] = silo.get('id')
        
        market1 = silo.get('market1', {})
        market2 = silo.get('market2', {})
        
        market1_id = market1.get('id')
        market2_id = market2.get('id')
        
        # Determine which market is debt and which is collateral
        if market_id == market1_id:
            debt_market = market1
            collateral_market = market2
        elif market_id == market2_id:
            debt_market = market2
            collateral_market = market1
        else:
            # Fallback if no match
            debt_market = market1
            collateral_market = market2
        
        # Process debt market (debt_asset and debt_token)
        if not is_collateral:
            result['debt_market_id'] = debt_market.get('id')
            result['debt_market_name'] = debt_market.get('name')
            
            # debt_asset (from inputToken)
            input_token = debt_market.get('inputToken', {})
            result['debt_asset_id'] = input_token.get('id')
            result['debt_asset_symbol'] = input_token.get('symbol')
            result['debt_asset_decimals'] = input_token.get('decimals')
            
            # debt_token (from dToken)
            d_token = debt_market.get('dToken', {})
            result['debt_token_id'] = d_token.get('id')
            result['debt_token_symbol'] = d_token.get('symbol')
            result['debt_token_decimals'] = d_token.get('decimals')
        
        # Process collateral market (collateral_asset and collateral_token)
        if is_collateral:
            result['collateral_market_id'] = debt_market.get('id')
            result['collateral_market_name'] = debt_market.get('name')
            
            # collateral_asset (from inputToken)
            input_token = debt_market.get('inputToken', {})
            result['collateral_asset_id'] = input_token.get('id')
            result['collateral_asset_symbol'] = input_token.get('symbol')
            result['collateral_asset_decimals'] = input_token.get('decimals')
            
            # collateral_token (from spToken)
            sp_token = debt_market.get('spToken', {})
            result['collateral_token_id'] = sp_token.get('id')
            result['collateral_token_symbol'] = sp_token.get('symbol')
            result['collateral_token_decimals'] = sp_token.get('decimals')

            s_token = debt_market.get('sToken', {})
            result['s_token_decimals'] = s_token.get("decimals")
        
        return result
    
    # Extract the entity data
    if 'data' in data and entity_key in data['data']:
        entities = data['data'][entity_key]
    else:
        entities = data.get(entity_key, [])
    
    # Process each position
    processed_data = [process_position(entity) for entity in entities]
    
    # Create DataFrame
    df = pd.DataFrame(processed_data)
    
    # Convert numeric columns
    for col in df.columns:
        df[col] = df[col].apply(convert_to_numeric)
    
    df = df.infer_objects()
    
    return df



In [255]:
# RPC CALLS FUNCTIONS

import nest_asyncio
nest_asyncio.apply()

from multicall import Call, Multicall
from web3 import Web3
from typing import List

w3 = Web3(Web3.HTTPProvider(RPC_URL))

def get_token_balances(tokens: List[str], accounts: List[str]):
    calls = []
    for token, account in zip(tokens, accounts):
        token_address = Web3.to_checksum_address(token)
        account_address = Web3.to_checksum_address(account)
        
        call = Call(
            token_address,
            ['balanceOf(address)(uint256)', account_address],
            [[f'{token}-{account}', None]]
        )
        calls.append(call)
    
    multi = Multicall(calls, _w3=w3, gas_limit=1_000_000, multicall_address="0xcA11bde05977b3631167028862bE2a173976CA11")
    results = multi()
    print(results)
    
    results_list = []
    for token, account in zip(tokens, accounts):
        key = f'{token}-{account}'
        try:
            results_list.append(results[key])
        except:
            results_list.append(0)
    
    return results_list


def get_ltvs(silos: List[str], accounts: List[str], silo_lens: str = SILO_LENS):
    calls = []
    for silo, account in zip(silos, accounts):
        if (silo in ["0x2433d6ac11193b4695d9ca73530de93c538ad18a","0x9d89cf7636a11b209865d99792d9e20a5e8bef21","0xb1412442aa998950f2f652667d5eba35fe66e43f"]):
            continue
        call = Call(
            silo_lens,
            ['getUserLTV(address,address)(uint256)', silo, account],
            [[f'{silo}-{account}', None]]
        )
        calls.append(call)
        print(silo,account)
    
    multi = Multicall(calls, _w3=w3, gas_limit=5_000_000)
    # multi.multicall_address = "0xcA11bde05977b3631167028862bE2a173976CA11"
    results = multi()
    
    results_list = []
    for silo, account in zip(silos, accounts):
        key = f'{silo}-{account}'
        try:
            results_list.append(results[key])
        except:
            results_list.append(0)
    
    return results_list

def get_lts(silos: List[str], silo_lens: str = SILO_LENS):
    calls = []
    for silo in silos:
        call = Call(
            silo_lens,
            ['getLt(address)(uint256)', silo],
            [[silo, None]]
        )
        calls.append(call)
    
    multi = Multicall(calls, _w3=w3, gas_limit=5_000_000)
    results = multi()
    
    results_list = []
    for silo in silos:
        try:
            results_list.append(results[silo])
        except:
            results_list.append(0)
    
    return results_list

def convert_shares_to_assets(silos: List[str], shares_list: List[int], is_debt: int):
    calls = []
    for i, [silo, shares] in enumerate(zip(silos, shares_list)):
        call = Call(
            silo,
            ['convertToAssets(uint256,uint8)(uint256)', shares, is_debt],
            [[str(i), None]]
        )
        calls.append(call)
    
    multi = Multicall(calls, _w3=w3, gas_limit=5_000_000)
    results = multi()
    
    results_list = []
    for i in range(len(silos)):
        try:
            results_list.append(results[str(i)])
        except:
            results_list.append(0)
    
    return results_list

def get_borrower_collateral_silos(silos: List[str], accounts: List[str]):
    calls = []
    for silo, account in zip(silos, accounts):
        call = Call(
            silo,
            ['borrowerCollateralSilo(address)(address)', account],
            [[f'{silo}-{account}', None]]
        )
        calls.append(call)
    
    multi = Multicall(calls, _w3=w3, gas_limit=5_000_000)
    results = multi()
    
    results_list = []
    for silo, account in zip(silos, accounts):
        key = f'{silo}-{account}'
        try:
            results_list.append(results[key])
        except:
            results_list.append(0)
    
    return results_list


In [256]:
# BATCH RPC CALLS FUNCTIONS

from typing import List
import pandas as pd

def get_batched_ltvs(market_ids: List[str], account_ids: List[str], batch_size: int = 50):
    all_ltvs = []
    total_items = len(market_ids)
    
    for i in range(0, total_items, batch_size):
        batch_end = min(i + batch_size, total_items)
        
        print(f"Processing batch {i//batch_size + 1}: items {i} to {batch_end-1} ({batch_end-i} items)")
        
        market_batch = market_ids[i:batch_end]
        account_batch = account_ids[i:batch_end]
        
        batch_ltvs = get_ltvs(market_batch, account_batch)
        all_ltvs.extend(batch_ltvs)
    
    print(f"Completed: {len(all_ltvs)} total LTVs retrieved")
    return all_ltvs

def get_batched_collateral_silos(market_ids: List[str], account_ids: List[str], batch_size: int = 50):
    all_collateral_silos = []
    total_items = len(market_ids)
    
    for i in range(0, total_items, batch_size):
        batch_end = min(i + batch_size, total_items)
        
        print(f"Processing batch {i//batch_size + 1}: items {i} to {batch_end-1} ({batch_end-i} items)")
        
        market_batch = market_ids[i:batch_end]
        account_batch = account_ids[i:batch_end]
        
        batch_collateral_silos = get_borrower_collateral_silos(market_batch, account_batch)
        all_collateral_silos.extend(batch_collateral_silos)
    
    print(f"Completed: {len(all_collateral_silos)} total borrower silos retrieved")
    return all_collateral_silos

def convert_shares_to_assets_batched(
    silos: List[str],
    shares_list: List[int],
    is_debt: int,
    batch_size: int = 50
):
    all_assets = []
    total_items = len(silos)
    
    # Process in batches
    for i in range(0, total_items, batch_size):
        batch_end = min(i + batch_size, total_items)
        
        print(f"Processing batch {i//batch_size + 1}: items {i} to {batch_end-1} ({batch_end-i} items)")
        
        # Get batch slices
        silos_batch = silos[i:batch_end]
        shares_batch = shares_list[i:batch_end]
        
        # Call your function with the batch
        batch_assets = convert_shares_to_assets(silos_batch, shares_batch, is_debt)
        
        # Add to results
        all_assets.extend(batch_assets)
    
    print(f"Completed: {len(all_assets)} total assets retrieved")
    return all_assets

In [243]:
debt_data = {
    'data': {
        'positions': debt_positions
    }
}

df = flatten_graphql_to_dataframe(debt_data,False)
df

,id,account_id,d_token_balance,market_id,silo_id,debt_market_id,debt_market_name,debt_asset_id,debt_asset_symbol,debt_asset_decimals,debt_token_id,debt_token_symbol,debt_token_decimals
0,0x016c306e103fbf48ec24810d078c65ad13c5f11b-0x0...,0x00c4168dd6736e9850f50af8f3b6b5c113107e1a,2.197948e+06,0x016c306e103fbf48ec24810d078c65ad13c5f11b,0x6bdf0d12d4b534d5f46c53a90dddfbe6c0e85dc7,0x016c306e103fbf48ec24810d078c65ad13c5f11b,wS->wanS,0x039e2fb66102314ce7b64ce5ce3e5183bc94ad38,wS,18,0x3f5fa5642afea358d65f818102008f6f88bf5cae,dwS-25,18
1,0x016c306e103fbf48ec24810d078c65ad13c5f11b-0x0...,0x0304492c94d8dafab67756c7b7aaf94e2c9ca709,1.519464e+01,0x016c306e103fbf48ec24810d078c65ad13c5f11b,0x6bdf0d12d4b534d5f46c53a90dddfbe6c0e85dc7,0x016c306e103fbf48ec24810d078c65ad13c5f11b,wS->wanS,0x039e2fb66102314ce7b64ce5ce3e5183bc94ad38,wS,18,0x3f5fa5642afea358d65f818102008f6f88bf5cae,dwS-25,18
2,0x016c306e103fbf48ec24810d078c65ad13c5f11b-0x0...,0x0be722795469468e04ce572eb77eb5882c94d6f5,3.599761e+04,0x016c306e103fbf48ec24810d078c65ad13c5f11b,0x6bdf0d12d4b534d5f46c53a90dddfbe6c0e85dc7,0x016c306e103fbf48ec24810d078c65ad13c5f11b,wS->wanS,0x039e2fb66102314ce7b64ce5ce3e5183bc94ad38,wS,18,0x3f5fa5642afea358d65f818102008f6f88bf5cae,dwS-25,18
3,0x016c306e103fbf48ec24810d078c65ad13c5f11b-0x0...,0x0c0f0f033161d53e7cce53c1d99775237f88b465,4.292918e+03,0x016c306e103fbf48ec24810d078c65ad13c5f11b,0x6bdf0d12d4b534d5f46c53a90dddfbe6c0e85dc7,0x016c306e103fbf48ec24810d078c65ad13c5f11b,wS->wanS,0x039e2fb66102314ce7b64ce5ce3e5183bc94ad38,wS,18,0x3f5fa5642afea358d65f818102008f6f88bf5cae,dwS-25,18
4,0x016c306e103fbf48ec24810d078c65ad13c5f11b-0x0...,0x0d466c7ee02dcbf2f8604c2d7daeb62491917f52,2.605401e+04,0x016c306e103fbf48ec24810d078c65ad13c5f11b,0x6bdf0d12d4b534d5f46c53a90dddfbe6c0e85dc7,0x016c306e103fbf48ec24810d078c65ad13c5f11b,wS->wanS,0x039e2fb66102314ce7b64ce5ce3e5183bc94ad38,wS,18,0x3f5fa5642afea358d65f818102008f6f88bf5cae,dwS-25,18
...,...,...,...,...,...,...,...,...,...,...,...,...,...
2125,0xfbd3cccb196ce900be5e2d008a6c2fde90760408-0x0...,0x0ca6cadb6a8ca4199788b9a060c3285fe7b897fd,6.663138e+05,0xfbd3cccb196ce900be5e2d008a6c2fde90760408,0xed7f8c077711b86b574ed94bb84895fbf147cd8e,0xfbd3cccb196ce900be5e2d008a6c2fde90760408,wS->beS,0x039e2fb66102314ce7b64ce5ce3e5183bc94ad38,wS,18,0x69c986fb2a86b5b5371e51dbeb1261e686c96dce,dwS-52,18
2126,0xfbd3cccb196ce900be5e2d008a6c2fde90760408-0x4...,0x4efe2304c67324c772b733610ae76e2079e18075,5.188120e+04,0xfbd3cccb196ce900be5e2d008a6c2fde90760408,0xed7f8c077711b86b574ed94bb84895fbf147cd8e,0xfbd3cccb196ce900be5e2d008a6c2fde90760408,wS->beS,0x039e2fb66102314ce7b64ce5ce3e5183bc94ad38,wS,18,0x69c986fb2a86b5b5371e51dbeb1261e686c96dce,dwS-52,18
2127,0xfbd3cccb196ce900be5e2d008a6c2fde90760408-0xa...,0xa5b2c785ca398592dfe82f455764a3504d7ceb44,2.138936e+02,0xfbd3cccb196ce900be5e2d008a6c2fde90760408,0xed7f8c077711b86b574ed94bb84895fbf147cd8e,0xfbd3cccb196ce900be5e2d008a6c2fde90760408,wS->beS,0x039e2fb66102314ce7b64ce5ce3e5183bc94ad38,wS,18,0x69c986fb2a86b5b5371e51dbeb1261e686c96dce,dwS-52,18
2128,0xfbd3cccb196ce900be5e2d008a6c2fde90760408-0xd...,0xdad00eca971d7b22e0de1b874fbae30471b75354,2.309454e+03,0xfbd3cccb196ce900be5e2d008a6c2fde90760408,0xed7f8c077711b86b574ed94bb84895fbf147cd8e,0xfbd3cccb196ce900be5e2d008a6c2fde90760408,wS->beS,0x039e2fb66102314ce7b64ce5ce3e5183bc94ad38,wS,18,0x69c986fb2a86b5b5371e51dbeb1261e686c96dce,dwS-52,18


In [244]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2130 entries, 0 to 2129
Data columns (total 13 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   id                   2130 non-null   str    
 1   account_id           2130 non-null   str    
 2   d_token_balance      2130 non-null   float64
 3   market_id            2130 non-null   str    
 4   silo_id              2130 non-null   str    
 5   debt_market_id       2130 non-null   str    
 6   debt_market_name     2130 non-null   str    
 7   debt_asset_id        2130 non-null   str    
 8   debt_asset_symbol    2130 non-null   str    
 9   debt_asset_decimals  2130 non-null   int64  
 10  debt_token_id        2130 non-null   str    
 11  debt_token_symbol    2130 non-null   str    
 12  debt_token_decimals  2130 non-null   int64  
dtypes: float64(1), int64(2), str(10)
memory usage: 216.5 KB


In [245]:
# Get what are the collateral silos for each borrow

collateral_silos = get_batched_collateral_silos(df["silo_id"],df["account_id"],40)
df["collateral_market"] = collateral_silos

Processing batch 1: items 0 to 39 (40 items)
Processing batch 2: items 40 to 79 (40 items)
Processing batch 3: items 80 to 119 (40 items)
Processing batch 4: items 120 to 159 (40 items)
Processing batch 5: items 160 to 199 (40 items)
Processing batch 6: items 200 to 239 (40 items)
Processing batch 7: items 240 to 279 (40 items)
Processing batch 8: items 280 to 319 (40 items)
Processing batch 9: items 320 to 359 (40 items)
Processing batch 10: items 360 to 399 (40 items)
Processing batch 11: items 400 to 439 (40 items)
Processing batch 12: items 440 to 479 (40 items)
Processing batch 13: items 480 to 519 (40 items)
Processing batch 14: items 520 to 559 (40 items)
Processing batch 15: items 560 to 599 (40 items)
Processing batch 16: items 600 to 639 (40 items)
Processing batch 17: items 640 to 679 (40 items)
Processing batch 18: items 680 to 719 (40 items)
Processing batch 19: items 720 to 759 (40 items)
Processing batch 20: items 760 to 799 (40 items)
Processing batch 21: items 800 to 8

In [246]:
lender_positions_ids = []

for index,row in df.iterrows():
    account_id = row["account_id"]
    collateral_market = row["collateral_market"]
    id = collateral_market + "-" + account_id + "-LENDER"
    lender_positions_ids.append(id)

len(lender_positions_ids)
lender_positions_ids[0]

lender_positions = fetch_lender_positions(SUBGRAPH_URL, headers, lender_positions_ids)
print(f"\nSuccessfully fetched {len(lender_positions)} lender positions in total!")

Batch 1: Fetched 1000 positions (Total: 1000)
Batch 2: Fetched 1000 positions (Total: 2000)
Batch 3: Fetched 130 positions (Total: 2130)

Successfully fetched 2130 lender positions in total!


In [247]:
lender_data = {
    'data': {
        'positions': lender_positions
    }
}

In [248]:
supply = flatten_graphql_to_dataframe(lender_data,True)
supply = supply.drop(["id","market_id"],axis=1)
supply.info()

<class 'pandas.DataFrame'>
RangeIndex: 2130 entries, 0 to 2129
Data columns (total 13 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   account_id                 2130 non-null   str    
 1   s_token_balance            2130 non-null   float64
 2   sp_token_balance           2130 non-null   float64
 3   silo_id                    2130 non-null   str    
 4   collateral_market_id       2130 non-null   str    
 5   collateral_market_name     2130 non-null   str    
 6   collateral_asset_id        2130 non-null   str    
 7   collateral_asset_symbol    2130 non-null   str    
 8   collateral_asset_decimals  2130 non-null   int64  
 9   collateral_token_id        2130 non-null   str    
 10  collateral_token_symbol    2130 non-null   str    
 11  collateral_token_decimals  2130 non-null   int64  
 12  s_token_decimals           2130 non-null   int64  
dtypes: float64(2), int64(3), str(8)
memory usage: 216.5 KB


In [249]:
df = df.merge(supply,how="left",on=["account_id","silo_id"])
# df["sp_token_balance"] = df["sp_token_balance"].fillna(0.0)
# df["s_token_balance"] = df["s_token_balance"].fillna(0.0)
df = df.drop(["id"],axis=1)
df

,account_id,d_token_balance,market_id,silo_id,debt_market_id,debt_market_name,debt_asset_id,debt_asset_symbol,debt_asset_decimals,debt_token_id,...,sp_token_balance,collateral_market_id,collateral_market_name,collateral_asset_id,collateral_asset_symbol,collateral_asset_decimals,collateral_token_id,collateral_token_symbol,collateral_token_decimals,s_token_decimals
0,0x00c4168dd6736e9850f50af8f3b6b5c113107e1a,2.197948e+06,0x016c306e103fbf48ec24810d078c65ad13c5f11b,0x6bdf0d12d4b534d5f46c53a90dddfbe6c0e85dc7,0x016c306e103fbf48ec24810d078c65ad13c5f11b,wS->wanS,0x039e2fb66102314ce7b64ce5ce3e5183bc94ad38,wS,18,0x3f5fa5642afea358d65f818102008f6f88bf5cae,...,2.731513e+09,0x21580de05c4f3d6d6a5345b03a898c33b872ab51,wanS->wS,0xfa85fe5a8f5560e9039c04f2b0a90de1415abd70,wanS,18,0x42526fedd57706d73ebd3ffb0a7a299d6b36deb7,nbwanS-25,18,18
1,0x0304492c94d8dafab67756c7b7aaf94e2c9ca709,1.519464e+01,0x016c306e103fbf48ec24810d078c65ad13c5f11b,0x6bdf0d12d4b534d5f46c53a90dddfbe6c0e85dc7,0x016c306e103fbf48ec24810d078c65ad13c5f11b,wS->wanS,0x039e2fb66102314ce7b64ce5ce3e5183bc94ad38,wS,18,0x3f5fa5642afea358d65f818102008f6f88bf5cae,...,1.550826e+04,0x21580de05c4f3d6d6a5345b03a898c33b872ab51,wanS->wS,0xfa85fe5a8f5560e9039c04f2b0a90de1415abd70,wanS,18,0x42526fedd57706d73ebd3ffb0a7a299d6b36deb7,nbwanS-25,18,18
2,0x0be722795469468e04ce572eb77eb5882c94d6f5,3.599761e+04,0x016c306e103fbf48ec24810d078c65ad13c5f11b,0x6bdf0d12d4b534d5f46c53a90dddfbe6c0e85dc7,0x016c306e103fbf48ec24810d078c65ad13c5f11b,wS->wanS,0x039e2fb66102314ce7b64ce5ce3e5183bc94ad38,wS,18,0x3f5fa5642afea358d65f818102008f6f88bf5cae,...,4.011998e+07,0x21580de05c4f3d6d6a5345b03a898c33b872ab51,wanS->wS,0xfa85fe5a8f5560e9039c04f2b0a90de1415abd70,wanS,18,0x42526fedd57706d73ebd3ffb0a7a299d6b36deb7,nbwanS-25,18,18
3,0x0c0f0f033161d53e7cce53c1d99775237f88b465,4.292918e+03,0x016c306e103fbf48ec24810d078c65ad13c5f11b,0x6bdf0d12d4b534d5f46c53a90dddfbe6c0e85dc7,0x016c306e103fbf48ec24810d078c65ad13c5f11b,wS->wanS,0x039e2fb66102314ce7b64ce5ce3e5183bc94ad38,wS,18,0x3f5fa5642afea358d65f818102008f6f88bf5cae,...,6.780761e+06,0x21580de05c4f3d6d6a5345b03a898c33b872ab51,wanS->wS,0xfa85fe5a8f5560e9039c04f2b0a90de1415abd70,wanS,18,0x42526fedd57706d73ebd3ffb0a7a299d6b36deb7,nbwanS-25,18,18
4,0x0d466c7ee02dcbf2f8604c2d7daeb62491917f52,2.605401e+04,0x016c306e103fbf48ec24810d078c65ad13c5f11b,0x6bdf0d12d4b534d5f46c53a90dddfbe6c0e85dc7,0x016c306e103fbf48ec24810d078c65ad13c5f11b,wS->wanS,0x039e2fb66102314ce7b64ce5ce3e5183bc94ad38,wS,18,0x3f5fa5642afea358d65f818102008f6f88bf5cae,...,3.402547e+07,0x21580de05c4f3d6d6a5345b03a898c33b872ab51,wanS->wS,0xfa85fe5a8f5560e9039c04f2b0a90de1415abd70,wanS,18,0x42526fedd57706d73ebd3ffb0a7a299d6b36deb7,nbwanS-25,18,18
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2125,0x0ca6cadb6a8ca4199788b9a060c3285fe7b897fd,6.663138e+05,0xfbd3cccb196ce900be5e2d008a6c2fde90760408,0xed7f8c077711b86b574ed94bb84895fbf147cd8e,0xfbd3cccb196ce900be5e2d008a6c2fde90760408,wS->beS,0x039e2fb66102314ce7b64ce5ce3e5183bc94ad38,wS,18,0x69c986fb2a86b5b5371e51dbeb1261e686c96dce,...,0.000000e+00,0x19926d2163fde0d77f7d50bb88701a6f51f45fab,beS->wS,0x871a101dcf22fe4fe37be7b654098c801cba1c88,beS,18,0x90ab736c529d9b980ed33206a066f23bddbca9b9,nbbeS-52,18,18
2126,0x4efe2304c67324c772b733610ae76e2079e18075,5.188120e+04,0xfbd3cccb196ce900be5e2d008a6c2fde90760408,0xed7f8c077711b86b574ed94bb84895fbf147cd8e,0xfbd3cccb196ce900be5e2d008a6c2fde90760408,wS->beS,0x039e2fb66102314ce7b64ce5ce3e5183bc94ad38,wS,18,0x69c986fb2a86b5b5371e51dbeb1261e686c96dce,...,0.000000e+00,0xfbd3cccb196ce900be5e2d008a6c2fde90760408,wS->beS,0x039e2fb66102314ce7b64ce5ce3e5183bc94ad38,wS,18,0x542f05fb915f201258955478496730f3fbb6141b,nbwS-52,18,18
2127,0xa5b2c785ca398592dfe82f455764a3504d7ceb44,2.138936e+02,0xfbd3cccb196ce900be5e2d008a6c2fde90760408,0xed7f8c077711b86b574ed94bb84895fbf147cd8e,0xfbd3cccb196ce900be5e2d008a6c2fde90760408,wS->beS,0x039e2fb66102314ce7b64ce5ce3e5183bc94ad38,wS,18,0x69c986fb2a86b5b5371e51dbeb1261e6

In [250]:
# BAD DEBT SILOS FOR SONIC
if FILE_NAME == "sonic_borrowers.csv":
  bad_debt_silos = [
    '0xccddbbbd1e36a6eda3a84cdcee2040a86225ba71',
    '0xed9777944a2fb32504a410d23f246463b3f40908',
    '0x0ab02dd08c1555d1a20c76a6ea30e3e36f3e06d4',
    '0x6e8c150224d6e9b646889b96eff6f7fd742e2c22',
    '0x1c1791911483e98875d162355fec47f37613f0fb',
    '0x8c98b43bf61f2b07c4d26f85732217948fca2a90'
  ]
else:
  bad_debt_silos = []
df = df.loc[~df["market_id"].isin(bad_debt_silos)]

In [257]:
ltvs = get_batched_ltvs(df["market_id"],df["account_id"],40)
print(ltvs)

Processing batch 1: items 0 to 39 (40 items)
0x016c306e103fbf48ec24810d078c65ad13c5f11b 0x00c4168dd6736e9850f50af8f3b6b5c113107e1a
0x016c306e103fbf48ec24810d078c65ad13c5f11b 0x0304492c94d8dafab67756c7b7aaf94e2c9ca709
0x016c306e103fbf48ec24810d078c65ad13c5f11b 0x0be722795469468e04ce572eb77eb5882c94d6f5
0x016c306e103fbf48ec24810d078c65ad13c5f11b 0x0c0f0f033161d53e7cce53c1d99775237f88b465
0x016c306e103fbf48ec24810d078c65ad13c5f11b 0x0d466c7ee02dcbf2f8604c2d7daeb62491917f52
0x016c306e103fbf48ec24810d078c65ad13c5f11b 0x0db87337abd3bcc447dd981640c5613e38a734ef
0x016c306e103fbf48ec24810d078c65ad13c5f11b 0x0db94d7b1111092c94ded5a5d99889c046fb5ac1
0x016c306e103fbf48ec24810d078c65ad13c5f11b 0x13b3531328734e34b4fd446d2f82ceacffef3118
0x016c306e103fbf48ec24810d078c65ad13c5f11b 0x1696d1d70e395c5bf43c106d3f77737365a742cf
0x016c306e103fbf48ec24810d078c65ad13c5f11b 0x1caa7c464006f4d9d2b5db9b0384c0f3edfa7a8f
0x016c306e103fbf48ec24810d078c65ad13c5f11b 0x1e55aae913d4de20a1ec5e07d83c87db6b0349c7
0x016c306

In [258]:
debt_shares_units = [int(balance * 10 ** decimals) for balance, decimals in zip(df["d_token_balance"].to_list(),df["debt_token_decimals"].to_list())]
debt_assets = convert_shares_to_assets_batched(df["market_id"], debt_shares_units, 2, 40)
print(debt_assets)

Processing batch 1: items 0 to 39 (40 items)
Processing batch 2: items 40 to 79 (40 items)
Processing batch 3: items 80 to 119 (40 items)
Processing batch 4: items 120 to 159 (40 items)
Processing batch 5: items 160 to 199 (40 items)
Processing batch 6: items 200 to 239 (40 items)
Processing batch 7: items 240 to 279 (40 items)
Processing batch 8: items 280 to 319 (40 items)
Processing batch 9: items 320 to 359 (40 items)
Processing batch 10: items 360 to 399 (40 items)
Processing batch 11: items 400 to 439 (40 items)
Processing batch 12: items 440 to 479 (40 items)
Processing batch 13: items 480 to 519 (40 items)
Processing batch 14: items 520 to 559 (40 items)
Processing batch 15: items 560 to 599 (40 items)
Processing batch 16: items 600 to 639 (40 items)
Processing batch 17: items 640 to 679 (40 items)
Processing batch 18: items 680 to 719 (40 items)
Processing batch 19: items 720 to 759 (40 items)
Processing batch 20: items 760 to 799 (40 items)
Processing batch 21: items 800 to 8

In [259]:
df["debt_assets"] = debt_assets
df["debt_assets"] = df["debt_assets"] / 10**df["debt_asset_decimals"]
df["debt_assets"]

0       2319148.327304
1            16.032515
2         37982.609723
3          4529.641335
4         27490.701372
             ...      
2125     701789.230198
2126      54643.420099
2127         225.28153
2128       2432.411987
2129         512.20518
Name: debt_assets, Length: 2089, dtype: object

In [260]:
collateral_shares_units = [int(balance * 10 ** decimals) for balance, decimals in zip(df["sp_token_balance"].to_list(),df["collateral_token_decimals"].to_list())]
collateral_assets = convert_shares_to_assets_batched(df["collateral_market_id"], collateral_shares_units, 0, 40)
print(collateral_assets)

borrowable_shares_units = [int(balance * 10 ** decimals) for balance, decimals in zip(df["s_token_balance"].to_list(),df["s_token_decimals"].to_list())]
borrowable_assets = convert_shares_to_assets_batched(df["collateral_market_id"], borrowable_shares_units, 1, 40)
print(borrowable_assets)

df["s_assets"] = borrowable_assets
df["sp_assets"] = collateral_assets
df["collateral_assets"] =  (df["s_assets"] / 10**df["s_token_decimals"]) + (df["sp_assets"] / 10**df["collateral_asset_decimals"])
df["collateral_assets"]

Processing batch 1: items 0 to 39 (40 items)
Processing batch 2: items 40 to 79 (40 items)
Processing batch 3: items 80 to 119 (40 items)
Processing batch 4: items 120 to 159 (40 items)
Processing batch 5: items 160 to 199 (40 items)
Processing batch 6: items 200 to 239 (40 items)
Processing batch 7: items 240 to 279 (40 items)
Processing batch 8: items 280 to 319 (40 items)
Processing batch 9: items 320 to 359 (40 items)
Processing batch 10: items 360 to 399 (40 items)
Processing batch 11: items 400 to 439 (40 items)
Processing batch 12: items 440 to 479 (40 items)
Processing batch 13: items 480 to 519 (40 items)
Processing batch 14: items 520 to 559 (40 items)
Processing batch 15: items 560 to 599 (40 items)
Processing batch 16: items 600 to 639 (40 items)
Processing batch 17: items 640 to 679 (40 items)
Processing batch 18: items 680 to 719 (40 items)
Processing batch 19: items 720 to 759 (40 items)
Processing batch 20: items 760 to 799 (40 items)
Processing batch 21: items 800 to 8

0       2731513.437771
1            15.508256
2         40119.980687
3          6780.761436
4         34025.468622
             ...      
2125     759522.743564
2126      63823.992222
2127        278.520446
2128       3168.682794
2129        696.574708
Name: collateral_assets, Length: 2089, dtype: object

In [261]:
df["s_assets"] = borrowable_assets
df["sp_assets"] = collateral_assets
df["collateral_assets"] =  (df["s_assets"] / 10**df["s_token_decimals"]) + (df["sp_assets"] / 10**df["collateral_asset_decimals"])
df["collateral_assets"]

0       2731513.437771
1            15.508256
2         40119.980687
3          6780.761436
4         34025.468622
             ...      
2125     759522.743564
2126      63823.992222
2127        278.520446
2128       3168.682794
2129        696.574708
Name: collateral_assets, Length: 2089, dtype: object

In [262]:
lt_markets = df["collateral_market_id"].unique().tolist()
lts = get_lts(lt_markets)

lt_df = pd.DataFrame({
    'collateral_market_id': lt_markets,
    'lt': lts
})

lt_df
lt_df["lt"] = lt_df["lt"] / 10**18

if "lt" in df.columns:
    df = df.drop("lt",axis=1)
if "lt_x" in df.columns:
    df = df.drop("lt_x",axis=1)
if "lt_y" in df.columns:
    df = df.drop("lt_y",axis=1)
df = df.merge(lt_df,on="collateral_market_id")

lt_df

,collateral_market_id,lt
0,0x21580de05c4f3d6d6a5345b03a898c33b872ab51,0.95
1,0x4cbd1ae8f51243f34a52e122ac1fd7b65b2b7bdd,0.92
2,0xbe0d3c8801206cc9f35a6626f90ef9f4f2983a3d,0.95
3,0x0588651ee0b84b3ca8035a69d60ff18c0263df71,0.97
4,0x356fcc93b96c8590e02fd6077f8886e1b31e2122,0.92
...,...,...
59,0x11ba70c0ebab7946ac84f0e6d79162b0cbb2693f,0.75
60,0x854475b78880767e246163031b5be44f14426c26,0.90
61,0x558d6d6d53270ae8ba622daf123983d9f3c21792,0.95
62,0x7db82f430f333ac5d93963e0a93fafef7061f998,0.95


In [263]:

import numpy as np

df["ltv"] = ltvs
df["ltv"] = df["ltv"] / 10**18
df["health"] = df["lt"] / df["ltv"].replace(0, 0.0000001)
df

,account_id,d_token_balance,market_id,silo_id,debt_market_id,debt_market_name,debt_asset_id,debt_asset_symbol,debt_asset_decimals,debt_token_id,...,collateral_token_symbol,collateral_token_decimals,s_token_decimals,debt_assets,s_assets,sp_assets,collateral_assets,lt,ltv,health
0,0x00c4168dd6736e9850f50af8f3b6b5c113107e1a,2.197948e+06,0x016c306e103fbf48ec24810d078c65ad13c5f11b,0x6bdf0d12d4b534d5f46c53a90dddfbe6c0e85dc7,0x016c306e103fbf48ec24810d078c65ad13c5f11b,wS->wanS,0x039e2fb66102314ce7b64ce5ce3e5183bc94ad38,wS,18,0x3f5fa5642afea358d65f818102008f6f88bf5cae,...,nbwanS-25,18,18,2319148.327304,0,2731513437771302783766496,2731513.437771,0.95,0.773104,1.228812
1,0x0304492c94d8dafab67756c7b7aaf94e2c9ca709,1.519464e+01,0x016c306e103fbf48ec24810d078c65ad13c5f11b,0x6bdf0d12d4b534d5f46c53a90dddfbe6c0e85dc7,0x016c306e103fbf48ec24810d078c65ad13c5f11b,wS->wanS,0x039e2fb66102314ce7b64ce5ce3e5183bc94ad38,wS,18,0x3f5fa5642afea358d65f818102008f6f88bf5cae,...,nbwanS-25,18,18,16.032515,0,15508255898554125517,15.508256,0.95,0.941351,1.009188
2,0x0be722795469468e04ce572eb77eb5882c94d6f5,3.599761e+04,0x016c306e103fbf48ec24810d078c65ad13c5f11b,0x6bdf0d12d4b534d5f46c53a90dddfbe6c0e85dc7,0x016c306e103fbf48ec24810d078c65ad13c5f11b,wS->wanS,0x039e2fb66102314ce7b64ce5ce3e5183bc94ad38,wS,18,0x3f5fa5642afea358d65f818102008f6f88bf5cae,...,nbwanS-25,18,18,37982.609723,0,40119980686841616103637,40119.980687,0.95,0.862059,1.102012
3,0x0c0f0f033161d53e7cce53c1d99775237f88b465,4.292918e+03,0x016c306e103fbf48ec24810d078c65ad13c5f11b,0x6bdf0d12d4b534d5f46c53a90dddfbe6c0e85dc7,0x016c306e103fbf48ec24810d078c65ad13c5f11b,wS->wanS,0x039e2fb66102314ce7b64ce5ce3e5183bc94ad38,wS,18,0x3f5fa5642afea358d65f818102008f6f88bf5cae,...,nbwanS-25,18,18,4529.641335,0,6780761435919586135376,6780.761436,0.95,0.608273,1.561799
4,0x0d466c7ee02dcbf2f8604c2d7daeb62491917f52,2.605401e+04,0x016c306e103fbf48ec24810d078c65ad13c5f11b,0x6bdf0d12d4b534d5f46c53a90dddfbe6c0e85dc7,0x016c306e103fbf48ec24810d078c65ad13c5f11b,wS->wanS,0x039e2fb66102314ce7b64ce5ce3e5183bc94ad38,wS,18,0x3f5fa5642afea358d65f818102008f6f88bf5cae,...,nbwanS-25,18,18,27490.701372,0,34025468622207225784435,34025.468622,0.95,0.73569,1.291305
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2084,0x0ca6cadb6a8ca4199788b9a060c3285fe7b897fd,6.663138e+05,0xfbd3cccb196ce900be5e2d008a6c2fde90760408,0xed7f8c077711b86b574ed94bb84895fbf147cd8e,0xfbd3cccb196ce900be5e2d008a6c2fde90760408,wS->beS,0x039e2fb66102314ce7b64ce5ce3e5183bc94ad38,wS,18,0x69c986fb2a86b5b5371e51dbeb1261e686c96dce,...,nbbeS-52,18,18,701789.230198,759522743564289886376031,0,759522.743564,0.90,0.895327,1.005219
2085,0x4efe2304c67324c772b733610ae76e2079e18075,5.188120e+04,0xfbd3cccb196ce900be5e2d008a6c2fde90760408,0xed7f8c077711b86b574ed94bb84895fbf147cd8e,0xfbd3cccb196ce900be5e2d008a6c2fde90760408,wS->beS,0x039e2fb66102314ce7b64ce5ce3e5183bc94ad38,wS,18,0x69c986fb2a86b5b5371e51dbeb1261e686c96dce,...,nbwS-52,18,18,54643.420099,63823992221913420433639,0,63823.992222,0.90,0.856158,1.051208
2086,0xa5b2c785ca398592dfe82f455764a3504d7ceb44,2.138936e+02,0xfbd3cccb196ce900be5e2d008a6c2fde90760408,0xed7f8c077711b86b574ed94bb84895fbf147cd8e,0xfbd3cccb196ce900be5e2d008a6c2fde90760408,wS->beS,0x039e2fb66102314ce7b64ce5ce3e5183bc94ad38,wS,18,0x69c986fb2a86b5b5371e51dbeb1261e686c96dce,...,nbbeS-52,18,18,225.28153,278520446084336636844,0,278.520446,0.90,0.783762,1.148307
2087,0xdad00eca971d7b22e0de1b874fbae30471b75354,2.309454e+03,0xfbd3cccb196ce900be5e2d008a6c2fde90760408,0xed7f8c077711b86b574ed94bb84895fbf147cd8e,0xfbd3cccb196ce900be5e2d008a6c2fde90760408,wS->beS,0x039e2fb66102314ce7b64ce5ce3e5183bc94ad38,wS,18,0x69c986fb2a86b5b5371e51dbeb1261e686c96dce,...,nbbeS-52,18,18,2432.411987,3168682794030914062956,0,3168.682794,0.90,0.743831,1.209953


In [264]:
borrowers = df.rename(columns={
  "account_id":"account",
  "market_id":"debt_silo",
  "collateral_market_id":"collateral_silo",
})

borrowers = borrowers[["account","debt_silo","health","debt_assets","debt_asset_symbol","collateral_assets","collateral_asset_symbol","ltv","lt","collateral_silo"]]
borrowers = borrowers.sort_values("health").reset_index(drop=True)
borrowers.to_csv(FILE_NAME,index=False)

In [265]:
borrowers

,account,debt_silo,health,debt_assets,debt_asset_symbol,collateral_assets,collateral_asset_symbol,ltv,lt,collateral_silo
0,0x5a9f2ca69f82621c841efefabd1f244273cd0245,0x322e1d5384aa4ed66aeca770b95686271de61dc3,0.0,0.000015,USDC.e,0.0,wS,1157920892373162037076177353953865399186742400...,0.80,0xf55902de87bd80c6a35614b48d7f8b612a083c12
1,0x06168e4c2ad4c73c3b74415f3db8336d0cbf2067,0x322e1d5384aa4ed66aeca770b95686271de61dc3,0.0,0.000222,USDC.e,0.0,wS,1157920892373162037076177353953865399186742400...,0.80,0xf55902de87bd80c6a35614b48d7f8b612a083c12
2,0x9a06c0afe8feeeee3c9e70c5d914d9fd108a8482,0x427514a905fa6beaed9a36e308fcfa06ce54e95b,0.0,0.0,WETH,0.0,wS,1157920892373162037076177353953865399186742400...,0.85,0x4bfead9975a64545c3594090327ef6666c2f6164
3,0x878a4e1803a0f44fc21316538b0717d866a4ba08,0x427514a905fa6beaed9a36e308fcfa06ce54e95b,0.0,0.0,WETH,0.0,wS,1157920892373162037076177353953865399186742400...,0.85,0x4bfead9975a64545c3594090327ef6666c2f6164
4,0x750a908f89708d6aae2797df68bd6550bd58d16d,0x427514a905fa6beaed9a36e308fcfa06ce54e95b,0.0,0.0,WETH,0.0,wS,1157920892373162037076177353953865399186742400...,0.85,0x4bfead9975a64545c3594090327ef6666c2f6164
...,...,...,...,...,...,...,...,...,...,...
2084,0x8b23f05cb0f7265985b194cbe453dbb76e53db9a,0xf55902de87bd80c6a35614b48d7f8b612a083c12,49987496877.343559,0.0,wS,0.000001,USDC.e,0.0,0.80,0x322e1d5384aa4ed66aeca770b95686271de61dc3
2085,0x8a5bdcbaca0a8cf00365fce6532c7b81545f3c48,0xf55902de87bd80c6a35614b48d7f8b612a083c12,57128570919.51506,0.0,wS,0.000001,USDC.e,0.0,0.80,0x322e1d5384aa4ed66aeca770b95686271de61dc3
2086,0xf3c0baafc614f5a22868ac033335a5dc97052f39,0xf55902de87bd80c6a35614b48d7f8b612a083c12,57128570919.51506,0.0,wS,0.000001,USDC.e,0.0,0.80,0x322e1d5384aa4ed66aeca770b95686271de61dc3
2087,0xba54af640f0a4e392717dc83cfc3041fce756fba,0xf55902de87bd80c6a35614b48d7f8b612a083c12,61523076036.908005,0.0,wS,0.000001,USDC.e,0.0,0.80,0x322e1d5384aa4ed66aeca770b95686271de61dc3


### Merge all chains borrowers

In [ ]:
# sonic = pd.read_csv("sonic_borrowers.csv")
# sonic["chain"] = "sonic"
# mainnet = pd.read_csv("mainnet_borrowers.csv")
# mainnet["chain"] = "mainnet"
# arb = pd.read_csv("arb_borrowers.csv")
# arb["chain"] = "arbitrum"
# avax = pd.read_csv("avax_borrowers.csv")
# avax["chain"] = "avalanche"

# merged = pd.concat([sonic,mainnet,arb,avax]).sort_values("health").reset_index(drop=True)
# cols = ['chain'] + [col for col in merged.columns if col != 'chain']
# merged = merged[cols]
# merged.to_csv("silo_borrowers.csv",index=False)